In [1]:
# Colab only: clone the repo and move into the project root
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [2]:
#| output: false
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


# URL VAE

This notebook is the starting point for everything that works on the URL text: the URL floors, the character-level URL VAE and the URL supervised reference. The data, the splits and the evaluation functions are already imported, so the first cells below only need to be run, and the model sections are where the work goes.

What is ready:

- **The items table** (`items`), with one row per kept URL: the URL, label, split, scheme, URL length and the other URL features. It was built by `2_data_processing.ipynb`, which must be run first.
- **The checks.** `src/checks.py` tests the items table, the model input and the evaluation functions, and the notebook runs them at the points where they matter.
- **The model input.** `encode_urls` turns URLs into integer arrays of length 256, the same way for everyone.
- **The evaluation functions.** A model saves one anomaly score per item, and `save_and_evaluate` turns the saved scores into the project metrics (AUROC, AUPRC, recall at 1% and 5% false-positive rate). The score format is described at the top of `src/evaluate.py`.

The image models live in `3_image_vae.ipynb`. URL models are scored on every item of the split.

## 1. Data

The items table holds every kept URL, so URL models use all of it (no `fits_canvas` filter).

In [3]:
# Imports and settings. Add the deep learning library (torch) here when building the model, and add it to requirements.txt
import json

import numpy as np
import pandas as pd

from src import checks
from src.evaluate import save_and_evaluate
from src.loaders import MAX_URL_LEN, VOCAB_SIZE, decode_url, encode_urls
from src.prepare_data import load_items
from src.splits import SPLIT_NAMES

PROC_DIR = root / "data" / "processed"
RESULTS_DIR = root / "results"
SCOPE = "all"    # URL models are scored on every item of the split (see src/evaluate.py)
SEED = 0

In [4]:
# Load the items table and its summary, then split the items into the sets each model uses
items = load_items(PROC_DIR)
summary = json.loads((PROC_DIR / "items_summary.json").read_text())

benign_train = items[(items["split"] == "train") & (items["cls"] == "benign")]
malicious_train = items[(items["split"] == "train") & (items["cls"] == "malicious")]
validation = items[items["split"] == "validation"]
test = items[items["split"] == "test"]

pd.DataFrame({
    "benign": items[items["cls"] == "benign"].groupby("split").size(),
    "malicious": items[items["cls"] == "malicious"].groupby("split").size(),
}).loc[SPLIT_NAMES]

,benign,malicious
split,,
train,300969,403020
validation,64494,86361
test,64494,86362


**Who uses which rows**

| Model | Fitted on | Tuned on | Threshold set on | Scored on |
| --- | --- | --- | --- | --- |
| Floors and the VAE | benign train rows | validation rows | benign validation rows | validation and test rows |
| Supervised reference | benign and malicious train rows | validation rows | benign validation rows | validation and test rows |

The test rows are used once per model, after the configuration is fixed on validation. Malicious train rows are used only by the supervised reference.

In the code above, `benign_train` is what the floors and the VAE are fitted on, `malicious_train` is used only by the supervised reference, and `validation` and `test` are scored.

## Checks on the data

Before any model is built, the next cell runs a set of checks on the items table and prints one row per check. Each row is `pass` or `FAIL`, and the cell stops with an error if any check fails, so a broken table cannot reach a model by accident. The checks confirm that the table matches `items_summary.json` (the row counts and the data version, which everyone on the team should see as the same value), that every split has both classes, that no registered domain sits in two splits of the same class, and that the URL columns are consistent. For the URL models it also checks that the floor features (`url_len`, `has_scheme` and the rest) have no missing values. Rows marked `info` are measurements for the reader and cannot fail.

In [5]:
# Run the data checks on the items table (stops with an error if any check fails)
checks.data_checks(items, summary, SCOPE).report()

check,result,detail
row count matches items_summary.json,pass,"1,005,700 rows"
data version matches items_summary.json,pass,0781657d11c5. Everyone on the team should see the same value
item_id equals the row position,pass,
label matches the class folder,pass,
rows per split and class match items_summary.json,pass,"train: 300,969 benign, 403,020 malicious; validation: 64,494 benign, 86,361 malicious; test: 64,494 benign, 86,362 malicious"
every split has benign and malicious rows,pass,
no domain appears in two splits of the same class,pass,
malicious share of every split is within 1 point of the overall share,pass,"train 0.572, validation 0.572, test 0.572 (overall 0.572)"
"no empty URL, and url_len equals the URL length",pass,
truncated is true exactly for URLs longer than 256,pass,"5,742 truncated"


## 2. Look at the model input

A URL is read as 256 integers. Id 0 is padding, id 1 is the unknown token, and ids 2 to 96 are the 95 printable ASCII symbols. A URL longer than 256 characters is cut off, and a shorter one is padded with 0 at the end, so a model must mask the padding in its loss. No start or end token is added: a sequence model that needs them adds its own and says so here. The vocabulary size is `VOCAB_SIZE` (97).

In [6]:
# Show how three training URLs are encoded and decoded
sample = benign_train.sample(3, random_state=SEED)
ids = encode_urls(sample["url"])
print("array shape:", ids.shape, "| vocabulary size:", VOCAB_SIZE)
for url_text, row in zip(sample["url"], ids):
    print(url_text)
    print("  first 25 ids:", row[:25].tolist())
    print("  decoded:     ", decode_url(row))

array shape: (3, 256) | vocabulary size: 97
poetry-archive.com/g/gray_john.html
  first 25 ids: [82, 81, 71, 86, 84, 91, 15, 67, 84, 69, 74, 75, 88, 71, 16, 69, 81, 79, 17, 73, 17, 73, 84, 67, 91]
  decoded:      poetry-archive.com/g/gray_john.html
myspace.com/beatbeatwhisper
  first 25 ids: [79, 91, 85, 82, 67, 69, 71, 16, 69, 81, 79, 17, 68, 71, 67, 86, 68, 71, 67, 86, 89, 74, 75, 85, 82]
  decoded:      myspace.com/beatbeatwhisper
youtube.com/watch?v=RqbmUMLUQ7g
  first 25 ids: [91, 81, 87, 86, 87, 68, 71, 16, 69, 81, 79, 17, 89, 67, 86, 69, 74, 33, 88, 31, 52, 83, 68, 79, 55]
  decoded:      youtube.com/watch?v=RqbmUMLUQ7g


In [7]:
# Share of URLs cut off at 256 characters, per split and class. Report the malicious test share with the results
items.groupby(["split", "cls"])["truncated"].mean().unstack().loc[SPLIT_NAMES].round(4)

cls,benign,malicious
split,,
train,0.0004,0.0130
validation,0.0004,0.0017
test,0.0005,0.0019


In [8]:
# Test items per scheme and per URL-length bin, by class: the strata that URL results are reported in
test_counts = {}
for col in ["scheme", "url_len_bin"]:
    t = test.groupby([col, "cls"]).size().unstack(fill_value=0)
    t.index = [f"{col}: {i}" for i in t.index]
    test_counts[col] = t
pd.concat(test_counts.values())

cls,benign,malicious
scheme: http,5515,8036
scheme: https,2734,28079
scheme: none,56245,50240
scheme: other,0,7
url_len_bin: 1-32,29740,57446
url_len_bin: 129-256,615,1039
url_len_bin: 33-64,25014,23214
url_len_bin: 65-128,9091,4502
url_len_bin: >256,34,161


**Reading.** Some strata have very few benign test items (for example the longest URLs, and URLs with an IP host). A stratified AUROC needs both classes in enough numbers, so small strata are reported as counts only.

## Checks on the model input

The next cell encodes 3,000 random URLs and checks the arrays the model will receive: shape and type, ids inside the vocabulary, padding only at the end, the right number of real ids per URL (the URL length, cut at 256), that decoding gives the URL back (with `?` for unknown characters), that a character outside printable ASCII becomes the unknown token (id 1) and that a URL longer than 256 characters is cut off. It also reports how many ids in the sample are the unknown token.

In [9]:
# Run the model input checks on a random sample of URLs
checks.url_input_checks(items).report()

check,result,detail
"array shape is (n, 256) and the type is int16",pass,"(3000, 256), int16"
ids are between 0 and 96 (vocabulary size 97),pass,"smallest 0, largest 96"
padding (id 0) only comes after the last character,pass,
"each row has min(URL length, 256) real ids",pass,
"decode_url(encode_urls(url)) gives back the URL (cut at the maximum length, unknown characters as ?)",pass,
a URL with a character outside printable ASCII gets the unknown token (id 1),pass,20 URLs tried
a URL longer than 256 is cut off at 256,pass,20 URLs tried
unknown token share,info,0.0022% of the real ids in this sample


## 3. Floors

A floor is a deliberately simple detector that uses no URL pattern, only trivial properties of the URL. A VAE is only useful if it beats its floor. Each floor is a Gaussian fitted on `benign_train` only, and the score is the distance from it (`1_eda.ipynb` section 7):

- **Floor A** (headline) uses one feature, `url_len`, and the score is the absolute z-score.
- **Floor B** (reported alongside) uses `url_len`, `has_scheme`, `host_is_ip`, `n_host_labels`, `path_len` and `digit_ratio`, and the score is the Mahalanobis distance (a distance that accounts for how the features vary together). A small ridge is added to the covariance because `host_is_ip` is almost constant among benign rows.

In [10]:
# Floor features (columns of the items table). Fit on benign train rows only, then score validation and test rows
FLOOR_A_FEATURES = ["url_len"]
FLOOR_B_FEATURES = ["url_len", "has_scheme", "host_is_ip", "n_host_labels", "path_len", "digit_ratio"]
# 1. Fit: mean (and covariance for floor B, with a small ridge) of the benign_train features.
# 2. Score: absolute z-score for floor A, Mahalanobis distance for floor B, for every validation and test row.
# 3. Save and evaluate, one call per floor:
#    save_and_evaluate(RESULTS_DIR, items, SCOPE, "floorA_url", "abs_z", 0,
#                      validation=(validation["item_id"], val_scores), test=(test["item_id"], test_scores))
#    and the same with "floorB_url" and "mahalanobis".

## 4. Character-level URL VAE

A GRU encoder and an autoregressive GRU decoder with teacher forcing, trained with KL annealing, on `benign_train` only. Log the KL term every epoch to catch posterior collapse (the decoder ignoring the latent code). Tune on `validation` only, then train at least 5 seeds of the chosen setup. Padding is masked out of the loss. The scores to save (higher means more likely malicious, one value per item) are per character, so URL length does not enter through summation:

- `nll_per_char`: negative log-likelihood per character
- `neg_elbo_per_char`: negative ELBO per character

Save each with `save_and_evaluate(RESULTS_DIR, items, SCOPE, "url_vae", "<score name>", seed, ...)`.

In [11]:
# Build, train and score the URL VAE here

## 5. Supervised reference

The reference is the VAE's GRU encoder plus one linear layer, trained from scratch with labels on `benign_train` and `malicious_train` using binary cross-entropy. Reusing the encoder architecture matches the capacity, so a gap to the VAE does not come from network size. It is context only, not a contribution, because the project itself must be generative. Save its scores as model `url_supervised`, score `prob_malicious` (the predicted probability of malicious).

**Read the reference results with two caveats.**

1. **The training data differs.** The VAE and the floors train on benign train rows only. The reference trains on benign and malicious train rows (more than twice as many). The gap between the VAE and its reference therefore measures what labels and malicious examples add, and part of it can come from simply having more data. Word it that way, not as "supervision alone". One optional check is to also train a reference on a random subset the size of the benign train set.
2. **The reference can use the shortcuts directly.** A labeled model can learn that a missing scheme points to benign and that an IP host points to malicious (scheme alone has AUROC 0.77, and IP hosts are 99.5% to 100% malicious, see `1_eda.ipynb` section 5). A high reference score may therefore mostly reflect those shortcuts. Read the reference results next to floor B and within the scheme strata, and do not describe them as an upper bound on what the URL carries.

A third point applies to the VAE and the reference equally: tuning on validation AUROC uses validation labels for model selection.

In [12]:
# Build, train and score the supervised reference here

## 6. Evaluate

Every model, floor and supervised reference is evaluated with the same call, `save_and_evaluate`, once per score and per seed. The model never computes a metric itself. The call:

1. saves the validation and test scores to `results/<model>/<score>/seed<k>/validation.csv.gz` and `test.csv.gz`,
2. checks them against the items table (exactly the items of the split, one finite score each, higher meaning more likely malicious),
3. returns one row of metrics: AUROC, AUPRC, recall at 1% and 5% false-positive rate, and the share of benign test items actually flagged at those thresholds. The alarm threshold is set on the benign validation rows only.

This section first shows that the pipeline works, then shows how to use it.

### Check that the evaluation pipeline works

The next cell runs `checks.evaluation_checks` with made-up scores in a temporary folder, so nothing is written to `results/`. Random scores must give an AUROC near 0.5, an AUPRC near the malicious share and about 1% and 5% of benign test items flagged. Perfect scores must give 1 everywhere, reversed scores must give an AUROC of 0, and a score file with a missing item, an unknown item or a NaN must be refused. Once real models exist, this cell can stay as a regression check.

In [13]:
# Run the evaluation checks with made-up scores in a temporary folder
checks.evaluation_checks(items, SCOPE).report()

check,result,detail
random scores give AUROC near 0.5,pass,AUROC 0.5002
random scores give AUPRC near the malicious share,pass,"AUPRC 0.5726, malicious share 0.5725"
random scores flag about 1% and 5% of test benign items at the validation thresholds,pass,0.99% and 5.10%
"perfect scores give AUROC 1, AUPRC 1 and recall 1 with no false alarms",pass,"AUROC 1.000, AUPRC 1.000, recall at 1% FPR 1.000"
reversed scores give AUROC 0 (higher must mean more malicious),pass,AUROC 0.000
a validation file that misses one item is refused,pass,
a validation file with an item that is not in the split is refused,pass,
scores with a missing value (NaN) are refused,pass,


### How to evaluate a model

The arguments are the results folder, the items table, `SCOPE`, the model name (`url_vae`), the score name (`nll_per_char`), the seed, and a pair of (item ids, scores) for validation and for test. The example below runs the call for two seeds with random numbers standing in for a model, then stacks the rows into one table.

It saves to `results/temp_demo_results/`, a folder that only holds this demo, so it can be deleted at any time and rerunning the cell overwrites it. In a real run, replace the random numbers with the model's scores and use `RESULTS_DIR` instead of `DEMO_DIR`, so the files land in `results/<model>/<score>/seed<k>/`.

In [14]:
# Example of the evaluate step: stand-in scores for two seeds, saved under results/temp_demo_results/ (use RESULTS_DIR in a real run)
DEMO_DIR = RESULTS_DIR / "temp_demo_results"
rng = np.random.default_rng(SEED)
rows = []
for seed in range(2):                                       # one call per seed
    val_scores = rng.normal(size=len(validation))           # one score per validation item, higher means more likely malicious
    test_scores = rng.normal(size=len(test))                # one score per test item
    metrics = save_and_evaluate(DEMO_DIR, items, SCOPE, "url_vae", "nll_per_char", seed,
                                validation=(validation["item_id"], val_scores),
                                test=(test["item_id"], test_scores))
    rows.append(metrics)                                    # one row per run, with the model, score and seed in it
pd.concat(rows, ignore_index=True)

,model,score,seed,scope,n_benign,n_malicious,prevalence,auroc,auprc,validation_auroc,recall_at_1pct_fpr,test_fpr_at_1pct,recall_at_5pct_fpr,test_fpr_at_5pct
0,url_vae,nll_per_char,0,all,64494,86362,0.57248,0.500194,0.572592,0.498899,0.009808,0.009923,0.051064,0.050981
1,url_vae,nll_per_char,1,all,64494,86362,0.57248,0.499891,0.572012,0.499913,0.010167,0.011272,0.050844,0.051199
